# 🛡️ EPI Finder - Treinamento de Detector de Capacete com YOLOv8 (Fase 4)

Bem-vindo à **Fase 4** do projeto **EPI Finder**!

Neste notebook, realizamos o **treinamento do modelo de Visão Computacional** através de **Transfer Learning** utilizando a arquitetura **YOLOv8** (versão Nano - `yolov8n.pt`). O modelo aprenderá a detectar e distinguir duas classes fundamentais de conformidade:
- `0: head` (sem capacete / cabeça desprotegida — **infração**)
- `1: helmet` (com capacete de proteção — **seguro**)

### 🎯 Objetivos deste Notebook:
1. **Verificar Ambiente e Hardware:** Diagnosticar recursos disponíveis (CPU / GPU CUDA) e definir a estratégia de processamento.
2. **Auditar o Dataset:** Validar o `data/data.yaml` e as divisões (`train`, `valid`, `test`).
3. **Carregar Modelo Base:** Obter os pesos pré-treinados no dataset COCO (`yolov8n.pt`).
4. **Configurar Hiperparâmetros:** Definir épocas, tamanho de batch, resolução (`imgsz=640`) e otimizadores.
5. **Executar o Treinamento:** Acompanhar as perdas (`box_loss`, `cls_loss`, `dfl_loss`) e métricas de validação.
6. **Monitorar e Avaliar os Resultados:** Plotar curvas de treinamento (`results.png`), matriz de confusão e previsões visuais.
7. **Exportar os Melhores Pesos:** Salvar `best.pt` no diretório `models/` para utilização nas Fases 5 (Avaliação Detalhada) e 6 (Inferência em Produção).

## 1. Configuração do Ambiente e Diagnóstico de Hardware

Configuramos o diretório raiz do projeto no `sys.path` e inspecionamos o suporte a aceleração por hardware (CUDA vs CPU).

In [1]:
import os
import sys
import shutil
from pathlib import Path
import yaml
import torch
import ultralytics
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Garante que a raiz do projeto esteja no sys.path e seja o diretório de trabalho atual
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print(f"📁 Diretório raiz do projeto: {PROJECT_ROOT}")
print(f"🐍 PyTorch versão: {torch.__version__}")
print(f"🚀 Ultralytics versão: {ultralytics.__version__}")

# Verificação de hardware (CUDA / CPU)
has_cuda = torch.cuda.is_available()
if has_cuda:
    device_name = torch.cuda.get_device_name(0)
    selected_device = 0
    print(f"⚡ Aceleração por GPU ativa: {device_name}")
else:
    num_cpus = os.cpu_count() or 1
    selected_device = 'cpu'
    print(f"💻 Executando em CPU ({num_cpus} threads de CPU detectadas)")
    print("ℹ️ Dica: Treinamento em CPU leva mais tempo por época. Recomendamos começar com poucas épocas (ex: 3 a 5) para validação rápida!")

📁 Diretório raiz do projeto: /app
🐍 PyTorch versão: 2.13.0+cu130
🚀 Ultralytics versão: 8.4.132
💻 Executando em CPU (4 threads de CPU detectadas)
ℹ️ Dica: Treinamento em CPU leva mais tempo por época. Recomendamos começar com poucas épocas (ex: 3 a 5) para validação rápida!


## 2. Validação da Configuração do Dataset (`data/data.yaml`)

O arquivo `data/data.yaml` instrui o YOLOv8 sobre onde localizar as imagens e rótulos de treino, validação e teste, bem como os nomes das classes.

In [2]:
yaml_path = PROJECT_ROOT / 'data' / 'data.yaml'
assert yaml_path.exists(), f"Arquivo de configuração não encontrado em: {yaml_path}"

with open(yaml_path, 'r', encoding='utf-8') as f:
    data_config = yaml.safe_load(f)

print("📋 Configuração do data.yaml:")
print(yaml.dump(data_config, default_flow_style=False))

# Validação interna da Ultralytics para checar resolução de caminhos
dataset_info = check_det_dataset(str(yaml_path))
print("✅ Dataset validado com sucesso pela Ultralytics:")
print(f"   - Treino:     {dataset_info['train']}")
print(f"   - Validação:  {dataset_info['val']}")
print(f"   - Teste:      {dataset_info.get('test', 'Não definido')}")
print(f"   - Classes:    {dataset_info['names']}")

📋 Configuração do data.yaml:
names:
  0: head
  1: helmet
nc: 2
path: ../data/dataset
test: test/images
train: train/images
val: valid/images

✅ Dataset validado com sucesso pela Ultralytics:
   - Treino:     /app/data/dataset/train/images
   - Validação:  /app/data/dataset/valid/images
   - Teste:      /app/data/dataset/test/images
   - Classes:    {0: 'head', 1: 'helmet'}


## 3. Modelo Base e Transfer Learning (`yolov8n.pt`)

### Por que Transfer Learning?
Treinar uma rede neural de detecção do zero requer dezenas de milhares de imagens e dias de processamento. Ao utilizar o **YOLOv8n** pré-treinado no dataset COCO (80 classes):
- A rede já possui camadas convolucionais que extraem primitivas visuais (arestas, texturas, formas humanas, contornos).
- Apenas ajustamos as camadas finais para focar nas nossas duas classes (`head` e `helmet`).
- A convergência ocorre muito mais rápido e com altíssima acurácia mesmo em um dataset com cerca de 1.000 imagens.

In [3]:
# Carrega o modelo pré-treinado YOLOv8n (o download ocorrerá automaticamente na primeira execução)
base_model_name = 'yolov8n.pt'
model = YOLO(base_model_name)

print(f"✅ Modelo base '{base_model_name}' inicializado com sucesso!")
print(f"ℹ️ Tipo de tarefa: {model.task}")

✅ Modelo base 'yolov8n.pt' inicializado com sucesso!
ℹ️ Tipo de tarefa: detect


## 4. Configuração dos Hiperparâmetros de Treinamento

Abaixo definimos os hiperparâmetros recomendados:

| Parâmetro | Valor Padrão | Descrição |
|---|---|---|
| `data` | `data/data.yaml` | Caminho do arquivo de configuração do dataset |
| `epochs` | `50` (ou `3` para teste rápido) | Número total de passagens pelo dataset de treino |
| `imgsz` | `640` | Resolução de entrada das imagens redimensionadas |
| `batch` | `16` (ou `8`) | Quantidade de imagens processadas por lote |
| `device` | `0` (GPU) ou `'cpu'` | Dispositivo de hardware selecionado dinamicamente |
| `name` | `helmet_detector_exp1` | Nome da pasta onde os logs, pesos e gráficos serão salvos |
| `patience` | `15` | Early stopping (interrompe o treino se não houver melhora em 15 épocas) |
| `workers` | `4` | Número de subprocessos do DataLoader para carregar batches |

In [ ]:
# Ajuste o número de épocas conforme sua necessidade:
# - 3 a 5 épocas: Smoke test / teste rápido para validação de pipeline
# - 30 a 50 épocas: Treinamento completo para máxima performance
EPOCHS = 50
BATCH_SIZE = 16 if has_cuda else 8
IMG_SIZE = 640
EXPERIMENT_NAME = 'helmet_detector_exp1'

train_args = {
    'data': str(yaml_path),
    'epochs': EPOCHS,
    'imgsz': IMG_SIZE,
    'batch': BATCH_SIZE,
    'device': selected_device,
    'name': EXPERIMENT_NAME,
    'patience': 15,
    'save': True,
    'verbose': True,
    'exist_ok': True  # Permite sobrescrever ou retomar se já existir
}

print("⚙️ Parâmetros configurados para o treino:")
for k, v in train_args.items():
    print(f"   - {k}: {v}")

## 5. Execução do Treinamento

A execução abaixo dispara o ciclo de treinamento do YOLOv8.
Durante as épocas, acompanhe no log:
- `box_loss`: erro de regressão das coordenadas delimitadoras da caixa.
- `cls_loss`: erro de classificação binária (`head` vs `helmet`).
- `dfl_loss`: erro da distribuição focal de fronteiras.
- `mAP50` e `mAP50-95`: métricas de acurácia média calculadas no conjunto de validação ao fim de cada época.

In [ ]:
print(f"🚀 Iniciando treinamento do experimento '{EXPERIMENT_NAME}'...")
training_results = model.train(**train_args)
print("🎉 Treinamento concluído com sucesso!")

## 6. Monitoramento de Perdas e Avaliação Visual dos Resultados

O Ultralytics salva automaticamente o histórico de treinamento, tabelas de métricas e gráficos ilustrativos na pasta `runs/detect/<EXPERIMENT_NAME>/`.

In [ ]:
exp_dir = PROJECT_ROOT / 'runs' / 'detect' / EXPERIMENT_NAME
csv_path = exp_dir / 'results.csv'

if csv_path.exists():
    df_results = pd.read_csv(csv_path)
    df_results.columns = [c.strip() for c in df_results.columns]
    
    print(f"📊 Total de épocas registradas: {len(df_results)}")
    print("\nÚltimas 5 épocas do treino:")
    display_cols = [c for c in df_results.columns if any(m in c for m in ['epoch', 'train/box_loss', 'train/cls_loss', 'val/box_loss', 'val/cls_loss', 'metrics/mAP50'])]
    display(df_results[display_cols].tail(5))
    
    # Melhor época com base no mAP50 de validação
    map50_col = 'metrics/mAP50(B)' if 'metrics/mAP50(B)' in df_results.columns else 'metrics/mAP50'
    if map50_col in df_results.columns:
        best_idx = df_results[map50_col].idxmax()
        best_epoch = df_results.loc[best_idx, 'epoch']
        best_score = df_results.loc[best_idx, map50_col]
        print(f"\n🏆 Melhor época: {int(best_epoch)} com mAP@50 = {best_score:.4f}")
else:
    print(f"Aviso: Arquivo {csv_path} não encontrado. O treino foi executado?")

### Gráficos Gerados pelo YOLOv8 (`results.png` e Matriz de Confusão)

Vamos exibir as imagens de diagnóstico geradas pelo treino.

In [ ]:
plots_to_show = [
    ('Gráfico Geral de Métricas e Perdas', exp_dir / 'results.png'),
    ('Matriz de Confusão', exp_dir / 'confusion_matrix.png'),
    ('Curva F1-Confidence', exp_dir / 'F1_curve.png'),
    ('Curva Precision-Recall (PR)', exp_dir / 'PR_curve.png')
]

for title, img_path in plots_to_show:
    if img_path.exists():
        print(f"\n🖼️ {title} ({img_path.name}):")
        img = Image.open(img_path)
        plt.figure(figsize=(12, 6))
        plt.imshow(img)
        plt.axis('off')
        plt.title(title, fontsize=14, pad=10)
        plt.show()
    else:
        print(f"ℹ️ Gráfico {img_path.name} ainda não disponível.")

## 7. Organização e Exportação dos Pesos (`models/`)

Copiamos os melhores pesos obtidos no treino (`best.pt`) para a pasta `models/` do projeto, deixando-os organizados para inferência e deploy.

In [ ]:
trained_best_weights = exp_dir / 'weights' / 'best.pt'
destination_dir = PROJECT_ROOT / 'models'
destination_dir.mkdir(parents=True, exist_ok=True)
destination_best = destination_dir / 'best.pt'
destination_named = destination_dir / f"{EXPERIMENT_NAME}_best.pt"

if trained_best_weights.exists():
    shutil.copy2(trained_best_weights, destination_best)
    shutil.copy2(trained_best_weights, destination_named)
    print(f"✅ Melhores pesos salvos com sucesso em:")
    print(f"   - {destination_best}")
    print(f"   - {destination_named}")
else:
    print(f"Aviso: Pesos de treinamento não localizados em {trained_best_weights}")

## 8. Teste Rápido de Predição com o Modelo Treinado

Executamos uma predição em uma amostra de teste para comprovar o funcionamento dos pesos recém-treinados.

In [ ]:
import glob

test_images = sorted(glob.glob(str(PROJECT_ROOT / 'data' / 'dataset' / 'test' / 'images' / '*.jpg')))

if test_images and destination_best.exists():
    sample_img_path = test_images[0]
    print(f"🔍 Testando predição na imagem de teste: {Path(sample_img_path).name}")
    
    best_model = YOLO(str(destination_best))
    pred_results = best_model.predict(source=sample_img_path, conf=0.25, save=False)
    
    # Visualização inline usando o método plot() do Ultralytics
    for r in pred_results:
        im_bgr = r.plot()  # Retorna array numpy BGR com bounding boxes desenhadas
        im_rgb = im_bgr[:, :, ::-1]  # Converte BGR para RGB com slicing NumPy
        
        plt.figure(figsize=(10, 8))
        plt.imshow(im_rgb)
        plt.axis('off')
        plt.title(f"Detecções do Modelo Treinado: {Path(sample_img_path).name}", fontsize=13)
        plt.show()
        
        print(f"Detecções encontradas: {len(r.boxes)}")
        for box in r.boxes:
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            cls_name = best_model.names[cls_id]
            print(f" - Classe: {cls_name} ({cls_id}) | Confiança: {conf:.2%}")
else:
    print("ℹ️ Execute o treinamento primeiro para testar a inferência com os pesos 'best.pt'.")

## 9. Próximos Passos (Fase 5 e Fase 6)

- **Fase 5 (Avaliação & Métricas):** Análise detalhada no conjunto de teste (`test split`), matriz de confusão normalizada, análise aprofundada de falsos positivos/falsos negativos com Pandas.
- **Fase 6 (Aplicação de Inferência):** Construção de `src/inference.py` com suporte a vídeos contínuos de câmeras de segurança, caixas verdes/vermelhas e geração de relatórios de auditoria.